# Zehirli / Zehirsiz Böcek Sınıflandırması — Omurga Karşılaştırması

Bu notebook, iNaturalist'ten toplanan iki veri setini (*Zehirli Bocek Veri Toplama* ve *Zehirli Bocek Ek Veri*) birleştirip iki farklı yaklaşımı eğitir ve karşılaştırır:

| | Yaklaşım | Çıktı |
|---|---|---|
| **Model A** | Doğrudan ikili sınıflandırma | zehirli / zehirsiz |
| **Model B** | Hiyerarşik: önce 38 takson grubu, sonra gruptan zehir durumu | grup → zehirli / zehirsiz |

**Model:** ImageNet üzerinde önceden eğitilmiş **EfficientNet-B0** (transfer öğrenme), 224×224 giriş.

**Veri:** 38 gruptan her birinden 190 fotoğraf (toplam ~7.200) doğrudan iNaturalist'ten çekilip 224×224 olarak belleğe alınır; veri artırma (yatay çevirme, rastgele kırpma/yakınlaştırma, parlaklık ve kontrast) GPU üzerinde yapılır.

**Veri bölme:**
- Her gruptaki türlerin bir kısmı **hiç eğitime girmez** (*görülmemiş tür testi*): modelin daha önce hiç görmediği türlerde zehirliliği ne kadar genelleyebildiğini ölçer.
- Kalan veri gruplara göre katmanlı olarak %80 eğitim, %10 doğrulama, %10 test şeklinde bölünür.

**Dengesizlik:** Ağırlıklı örnekleme: her sınıf (zehirli/zehirsiz) %50, sınıf içinde her grup eşit ağırlıkla örneklenir (oversampling).

**Değerlendirme:** Accuracy, F1, precision, zehirliler için recall (kaçırılan zehirli böcek tehlikelidir), ROC-AUC, karışıklık matrisleri, gruplara göre doğruluk, benzer görünümlü çiftler analizi ve Grad-CAM.

## 1. Kütüphaneler ve ayarlar

In [ ]:
import os, glob, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda"
EPOCH = 10
BATCH = 128
LR = 3e-4
IMG = 224
GRUP_BASINA = 190
CIKTI = "/kaggle/working"
print("GPU sayısı:", torch.cuda.device_count(), "| CPU:", os.cpu_count())

## 2. Fotoğrafları iNaturalist'ten çekme (her gruptan 190, toplam ~7.200)

Önceki notebook'larda (*Zehirli Bocek Veri Toplama* ve *Ek Veri*) tanımlanan 38 grubun her birinden 190 fotoğraf doğrudan iNaturalist'ten çekilir: Research Grade gözlemler, CC0 / CC-BY / CC-BY-NC lisanslı fotoğraflar, tırtıl gruplarında yalnızca larva evresi. Fotoğraflar 224×224'e getirilip doğrudan belleğe alınır.

In [ ]:
import io, requests

API = "https://api.inaturalist.org/v1"
LISANSLAR = ["cc0", "cc-by", "cc-by-nc"]
LARVA = {"term_id": 1, "term_value_id": 6}

GRUPLAR_TANIM = [
    # takson,          sınıf,      larva, hariç tutulan
    ("Vespidae", "zehirli", False, None), ("Apis mellifera", "zehirli", False, None),
    ("Bombus", "zehirli", False, None), ("Solenopsis", "zehirli", False, None),
    ("Pompilidae", "zehirli", False, None), ("Mutillidae", "zehirli", False, None),
    ("Meloidae", "zehirli", False, None), ("Paederus", "zehirli", False, None),
    ("Reduviidae", "zehirli", False, None), ("Thaumetopoea", "zehirli", True, None),
    ("Megalopygidae", "zehirli", True, None), ("Limacodidae", "zehirli", True, None),
    ("Pogonomyrmex", "zehirli", False, None), ("Hemileucinae", "zehirli", True, None),
    ("Lymantriinae", "zehirli", True, None),
    ("Syrphidae", "zehirsiz", False, None), ("Muscidae", "zehirsiz", False, None),
    ("Tipulidae", "zehirsiz", False, None), ("Pieridae", "zehirsiz", False, None),
    ("Lycaenidae", "zehirsiz", False, None), ("Odonata", "zehirsiz", False, None),
    ("Acrididae", "zehirsiz", False, None), ("Tettigoniidae", "zehirsiz", False, None),
    ("Mantodea", "zehirsiz", False, None), ("Cicadidae", "zehirsiz", False, None),
    ("Cerambycidae", "zehirsiz", False, None), ("Chrysopidae", "zehirsiz", False, None),
    ("Meliponini", "zehirsiz", False, None), ("Formicinae", "zehirsiz", False, None),
    ("Staphylinidae", "zehirsiz", False, "Paederus"), ("Notodontidae", "zehirsiz", True, "Thaumetopoeinae"),
    ("Saturniinae", "zehirsiz", True, None), ("Sphingidae", "zehirsiz", True, None),
    ("Tenthredinidae", "zehirsiz", False, None), ("Sesiidae", "zehirsiz", False, None),
    ("Pentatomidae", "zehirsiz", False, None), ("Coreidae", "zehirsiz", False, None),
    ("Chrysomelidae", "zehirsiz", False, None),
]

oturum = requests.Session()
oturum.headers["User-Agent"] = "zehirli-bocek-ders-projesi (Kaggle notebook)"

def api(yol, **params):
    for deneme in range(5):
        time.sleep(1.0)
        try:
            r = oturum.get(f"{API}/{yol}", params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(5 * (deneme + 1))
    raise RuntimeError(f"API isteği başarısız: {yol}")

def takson_id(ad):
    return next(t["id"] for t in api("taxa", q=ad, is_active="true", per_page=30)["results"] if t["name"] == ad)

satirlar = []
t0 = time.time()
for ad, sinif, larva, haric in GRUPLAR_TANIM:
    p = dict(taxon_id=takson_id(ad), quality_grade="research", photos="true",
             photo_license=",".join(LISANSLAR), per_page=200, order_by="id", order="desc")
    if haric:
        p["without_taxon_id"] = takson_id(haric)
    if larva:
        p.update(LARVA)
    n, sayfa = 0, 1
    while n < GRUP_BASINA and sayfa <= 5:
        sonuc = api("observations", page=sayfa, **p)["results"]
        if not sonuc:
            break
        for o in sonuc:
            foto = next((f for f in o["photos"] if f.get("license_code") in LISANSLAR), None)
            if foto is None or n >= GRUP_BASINA:
                continue
            satirlar.append(dict(sinif=sinif, grup=ad.replace(" ", "_"), photo_id=foto["id"],
                                 takson=o["taxon"]["name"], lisans=foto["license_code"], atif=foto.get("attribution"),
                                 url=foto["url"].replace("/square.", "/medium.")))
            n += 1
        sayfa += 1
    print(f"{ad:16s} {sinif:9s} {n} fotoğraf", flush=True)

veri = pd.DataFrame(satirlar).drop_duplicates("photo_id").reset_index(drop=True)
print(f"Liste hazır: {len(veri)} fotoğraf ({(time.time() - t0) / 60:.1f} dk)")

In [ ]:
GORUNTU = np.zeros((len(veri), IMG, IMG, 3), dtype=np.uint8)
basarili = np.zeros(len(veri), dtype=bool)

def indir(i):
    for deneme in range(3):
        try:
            r = requests.get(veri.url.iat[i], timeout=30)
            if r.status_code == 200:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                k = IMG / min(img.size)
                img = img.resize((max(IMG, round(img.width * k)), max(IMG, round(img.height * k))), Image.BILINEAR)
                sol, ust = (img.width - IMG) // 2, (img.height - IMG) // 2
                GORUNTU[i] = np.asarray(img.crop((sol, ust, sol + IMG, ust + IMG)))
                basarili[i] = True
                return
        except Exception:
            pass
        time.sleep(2)

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for j, _ in enumerate(ex.map(indir, range(len(veri))), 1):
        if j % 2000 == 0:
            print(f"{j}/{len(veri)} indirildi ({(time.time() - t0) / 60:.1f} dk)", flush=True)

GORUNTU = GORUNTU[basarili]
veri = veri[basarili].reset_index(drop=True)
GORUNTU_T = torch.from_numpy(GORUNTU)
veri.drop(columns="url").to_csv(f"{CIKTI}/metadata.csv", index=False)

veri["idx"] = np.arange(len(veri))
veri["etiket"] = (veri.sinif == "zehirli").astype(int)
GRUPLAR = sorted(veri.grup.unique())
grup_idx = {g: i for i, g in enumerate(GRUPLAR)}
veri["grup_id"] = veri.grup.map(grup_idx)
grup_sinif = veri.groupby("grup").sinif.first()
ZEHIRLI_GRUP_IDS = [grup_idx[g] for g in GRUPLAR if grup_sinif[g] == "zehirli"]

print(f"İndirme bitti: {len(veri)} fotoğraf | {(time.time() - t0) / 60:.1f} dk | {GORUNTU.nbytes / 1e9:.2f} GB")
print(f"{len(GRUPLAR)} grup | {veri.takson.nunique()} takson |", veri.sinif.value_counts().to_dict())

## 3. Eğitim / doğrulama / test ve görülmemiş tür bölmesi

In [ ]:
rng = np.random.default_rng(SEED)

# Görülmemiş tür testi: her grupta (en az 10 farklı taksonu olan) grubun ~%10'u kadar fotoğrafı
# oluşturan bazı taksonlar tamamen ayrılır. Tek başına grubun %5'inden büyük taksonlar ayrılmaz.
gorulmemis = []
for g, d in veri.groupby("grup"):
    say = d.takson.value_counts()
    if len(say) < 10:
        continue
    taksonlar = list(say.index)
    rng.shuffle(taksonlar)
    hedef, toplam = 0.10 * len(d), 0
    for t in taksonlar:
        if toplam >= hedef:
            break
        if say[t] > 0.05 * len(d):
            continue
        gorulmemis.append(t)
        toplam += say[t]

veri["bolum"] = "egitim"
veri.loc[veri.takson.isin(gorulmemis), "bolum"] = "gorulmemis_tur"

# Kalan veri: her grup içinde %80 eğitim, %10 doğrulama, %10 test
for g, idx in veri[veri.bolum == "egitim"].groupby("grup").groups.items():
    idx = rng.permutation(np.array(idx))
    n = len(idx)
    veri.loc[idx[: int(0.1 * n)], "bolum"] = "dogrulama"
    veri.loc[idx[int(0.1 * n): int(0.2 * n)], "bolum"] = "test"

veri.to_csv(f"{CIKTI}/veri_bolmesi.csv", index=False)
bol = {b: veri[veri.bolum == b].reset_index(drop=True) for b in ["egitim", "dogrulama", "test", "gorulmemis_tur"]}
print("Görülmemiş takson sayısı:", len(gorulmemis))
pd.crosstab(veri.bolum, veri.sinif, margins=True)

## 4. Veri artırma (GPU üzerinde) ve örnekleme

In [ ]:
ORT = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def batch_hazirla(idx, egitim):
    """Bellekteki fotoğraflardan bir batch oluşturur; eğitimde GPU üzerinde veri artırma uygular."""
    x = GORUNTU_T[torch.as_tensor(idx)].to(DEVICE).permute(0, 3, 1, 2).contiguous().float().div_(255)
    if egitim:
        cevir = torch.rand(len(x), device=DEVICE) < 0.5                 # yatay çevirme
        x[cevir] = x[cevir].flip(3)
        s = int(IMG * np.random.uniform(0.7, 1.0))                       # rastgele kırpma + yakınlaştırma
        i0, j0 = np.random.randint(0, IMG - s + 1, 2)
        x = F.interpolate(x[:, :, i0:i0 + s, j0:j0 + s], size=IMG, mode="bilinear", align_corners=False)
        parlaklik = 1 + (torch.rand(len(x), 1, 1, 1, device=DEVICE) - 0.5) * 0.4
        kontrast = 1 + (torch.rand(len(x), 1, 1, 1, device=DEVICE) - 0.5) * 0.4
        ort = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (((x - ort) * kontrast + ort) * parlaklik).clamp_(0, 1)     # renk tonu değiştirilmez: uyarı renkleri önemli
    return (x - ORT) / STD

tr = bol["egitim"]
TR_IDX, TR_E, TR_G = tr.idx.values, torch.tensor(tr.etiket.values), torch.tensor(tr.grup_id.values)

# Her sınıf %50, sınıf içinde her grup eşit olasılıkla örneklenir
grup_sayisi = tr.groupby("sinif").grup.nunique()
AGIRLIK = torch.tensor((1.0 / (tr.sinif.map(grup_sayisi) * tr.grup.map(tr.grup.value_counts()))).values)
ADIM = len(tr) // BATCH

def egitim_batchleri():
    secim = torch.multinomial(AGIRLIK, ADIM * BATCH, replacement=True).view(ADIM, BATCH)
    for s in secim:
        yield TR_IDX[s.numpy()], TR_E[s], TR_G[s]

def eval_batchleri(df, boyut=256):
    for i in range(0, len(df), boyut):
        yield df.idx.values[i:i + boyut]

print({b: len(d) for b, d in bol.items()}, "| epoch başına adım:", ADIM)

In [ ]:
# Veri artırma örnekleri
idx, e, g = next(egitim_batchleri())
x = (batch_hazirla(idx[:16], True) * STD + ORT).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax, xi, ei, gi in zip(axes.flat, x, e[:16], g[:16]):
    ax.imshow(xi)
    ax.set_title(f"{GRUPLAR[gi]}\n{'zehirli' if ei else 'zehirsiz'}", fontsize=8, color="red" if ei else "green")
    ax.axis("off")
plt.suptitle("Eğitim örnekleri (veri artırma uygulanmış)"); plt.tight_layout(); plt.show()

## 5. Model karşılaştırması (omurga seçimi)

Hiyerarşik yaklaşım (Model B: önce 38 grup, sonra zehir durumu) üç farklı önceden eğitilmiş omurgayla, **aynı veri ve aynı bölmeyle** eğitilir:

| Omurga | Parametre | Not |
|---|---|---|
| EfficientNet-B0 | ~5 M | önceki çalışmalarda kullanılan model |
| EfficientNet-B3 | ~12 M | aynı ailenin daha büyük sürümü |
| ConvNeXt-Tiny | ~28 M | daha modern evrişimli mimari |

Değerlendirmede TTA (normal + aynalanmış görüntü) kullanılır.

In [ ]:
OMURGALAR = {
    "EfficientNet-B0": (lambda: models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1), 3e-4),
    "EfficientNet-B3": (lambda: models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1), 2e-4),
    "ConvNeXt-Tiny":   (lambda: models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1), 1e-4),
}

def model_olustur(ad, n_cikis):
    m = OMURGALAR[ad][0]()
    if ad.startswith("EfficientNet"):
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_cikis)
    else:
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_cikis)
    return m

@torch.no_grad()
def tahmin_et(model, df):
    """TTA ile P(zehirli) ve grup olasılıkları."""
    model.eval()
    p_zehirli, p_grup = [], []
    for idx in eval_batchleri(df):
        with torch.autocast("cuda", dtype=torch.float16):
            x = batch_hazirla(idx, False)
            prob = (model(x).float().softmax(1) + model(x.flip(3)).float().softmax(1)) / 2
        p_grup.append(prob.cpu())
        p_zehirli.append(prob[:, ZEHIRLI_GRUP_IDS].sum(1).cpu())
    return torch.cat(p_zehirli).numpy(), torch.cat(p_grup).numpy()

def egit(ad):
    model = model_olustur(ad, len(GRUPLAR)).to(DEVICE)
    lr = OMURGALAR[ad][1]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCH * ADIM, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")
    kayip_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    y_val = bol["dogrulama"].etiket.values
    t_bas = time.time()
    for ep in range(1, EPOCH + 1):
        model.train()
        t0, toplam, n = time.time(), 0.0, 0
        for idx, e, g in egitim_batchleri():
            x = batch_hazirla(idx, True)
            with torch.autocast("cuda", dtype=torch.float16):
                kayip = kayip_fn(model(x), g.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(kayip).backward()
            scaler.step(opt)
            scaler.update()
            sch.step()
            toplam += kayip.item() * len(x)
            n += len(x)
        p, _ = tahmin_et(model, bol["dogrulama"])
        print(f"[{ad}] epoch {ep}/{EPOCH} | kayıp={toplam / n:.4f} | val F1={f1_score(y_val, p > 0.5):.4f} | "
              f"{n / (time.time() - t0):.0f} foto/sn", flush=True)
    return model, (time.time() - t_bas) / 60

## 6. Üç omurgayı eğitme

In [ ]:
def olc(y, p, esik=0.5):
    t = (p >= esik).astype(int)
    return dict(accuracy=accuracy_score(y, t), f1=f1_score(y, t), precision=precision_score(y, t, zero_division=0),
                recall_zehirli=recall_score(y, t), roc_auc=roc_auc_score(y, p))

sonuclar = []
for ad in OMURGALAR:
    torch.manual_seed(SEED); np.random.seed(SEED)
    model, sure = egit(ad)
    for b in ["test", "gorulmemis_tur"]:
        p, pg = tahmin_et(model, bol[b])
        sonuclar.append(dict(omurga=ad, bolum=b, egitim_dk=round(sure, 1),
                             grup_dogrulugu=(pg.argmax(1) == bol[b].grup_id.values).mean(),
                             **olc(bol[b].etiket.values, p)))
    torch.save(model.state_dict(), f"{CIKTI}/model_B_{ad}.pt")
    del model; torch.cuda.empty_cache()
    print(f"--- {ad} bitti ({sure:.1f} dk)", flush=True)

omurga_df = pd.DataFrame(sonuclar)
omurga_df.to_csv(f"{CIKTI}/omurga_karsilastirma.csv", index=False)
omurga_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, b in zip(axes, ["test", "gorulmemis_tur"]):
    d = omurga_df[omurga_df.bolum == b].set_index("omurga")[["accuracy", "f1", "precision", "recall_zehirli"]]
    d.plot.bar(ax=ax, rot=0)
    ax.set_title(f"Omurga karşılaştırması — {b}"); ax.set_ylim(0, 1); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print("=" * 80)
print("OMURGA KARŞILAŞTIRMASI (Model B, TTA)")
print("=" * 80)
print(omurga_df[["omurga", "bolum", "accuracy", "f1", "precision", "recall_zehirli", "roc_auc", "egitim_dk"]]
      .round(3).to_string(index=False))